In [ ]:
%%capture
!pip install stable-baselines3[extra]
!pip install moviepy
!pip install wandb -q
!pip install gymnasium[atari] ale-py

In [ ]:
# ---------------------------------------------------------------------------
# Mount Google Drive
# ---------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""
Dueling Double DQN — trained from scratch on Colab

Two improvements combined:

1. DOUBLE DQN
   Online net selects the next action, target net evaluates it:
       a* = argmax_a Q_online(s', a)
       target = r + γ * Q_target(s', a*)
   Removes Q-value overestimation bias.

2. DUELING ARCHITECTURE
   The Q-network splits into two streams after the CNN backbone:
       V(s)     — scalar state value: how good is this state overall?
       A(s, a)  — advantage per action: how much better is action a vs average?
   Combined as:
       Q(s, a) = V(s) + A(s, a) - mean_a(A(s, a))

Checkpoints are saved to Google Drive every CHECKPOINT_FREQ steps.
If the run disconnects, re-run all cells — it will auto-resume from the
latest checkpoint found in SAVE_DIR.

To run on Colab:
  1. Runtime > Change runtime type > T4 GPU
  2. Set SAVE_DIR below to a folder in your Drive
  3. Run all cells
  4. Enter your W&B API key when prompted
"""

import os
import glob
import numpy as np
import torch as th
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import ale_py
import wandb

from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback, CallbackList, CheckpointCallback
from stable_baselines3.common.logger import Video, HParam, TensorBoardOutputFormat
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.dqn.policies import CnnPolicy
from stable_baselines3.dqn.policies import QNetwork

from typing import Any, Dict

gym.register_envs(ale_py)

# ---------------------------------------------------------------------------
# CONFIGURE THIS
# ---------------------------------------------------------------------------

# Folder in your Google Drive where checkpoints and outputs are saved.
# Will be created if it doesn't exist.
SAVE_DIR = "/content/drive/MyDrive/Colab Notebooks/pacman/dueling_double_dqn_v1"

# Save a checkpoint every this many steps
CHECKPOINT_FREQ = 50_000

# ---------------------------------------------------------------------------

os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_FILE_NAME  = os.path.join(SAVE_DIR, "ALE-Pacman-v5")
BUFFER_FILE_NAME = os.path.join(SAVE_DIR, "dqn_replay_buffer_dueling_double_v1")
POLICY_FILE_NAME = os.path.join(SAVE_DIR, "dqn_policy_dueling_double_v1")

# =====Model Config=====
EVAL_CALLBACK_FREQ  = 50_000
VIDEO_CALLBACK_FREQ = 750_000
FRAMESKIP           = 4
NUM_TIMESTEPS       = 3_000_000

# =====Hyperparams (match dqn_v2-8 for a fair comparison)=====
EXPLORATION_FRACTION  = 0.3
BUFFER_SIZE           = 70_000
BATCH_SIZE            = 64
LEARNING_STARTS       = 100_000
LEARNING_RATE         = 0.00005
GAMMA                 = 0.999
FINAL_EPSILON         = 0.005
TARGET_UPDATE_INTERVAL = 1_000

In [ ]:
# ---------------------------------------------------------------------------
# Dueling Q-Network
# Subclasses SB3's QNetwork so all internal SB3 methods (_predict,
# set_training_mode, etc.) are inherited automatically.
# Only forward() is overridden to add the V + A dueling head.
# ---------------------------------------------------------------------------

class DuelingQNetwork(QNetwork):
    """
    Standard Q-net:  CNN -> Linear(features_dim, n_actions)

    Dueling Q-net:   CNN -> value_stream     -> Linear(64, 1)         = V(s)
                         -> advantage_stream -> Linear(64, n_actions) = A(s,a)
                     Q(s,a) = V(s) + A(s,a) - mean_a(A(s,a))

    The mean subtraction keeps V and A identifiable — without it V could
    absorb everything and A would be meaningless.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        features_dim = self.features_dim
        n_actions    = self.action_space.n

        self.value_stream = nn.Sequential(
            nn.Linear(features_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )
        self.advantage_stream = nn.Sequential(
            nn.Linear(features_dim, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions),
        )
        # Remove the standard single-head q_net built by QNetwork
        del self.q_net

    def forward(self, obs: th.Tensor) -> th.Tensor:
        features  = self.extract_features(obs, self.features_extractor)
        value     = self.value_stream(features)      # (batch, 1)
        advantage = self.advantage_stream(features)  # (batch, n_actions)
        return value + advantage - advantage.mean(dim=1, keepdim=True)


class DuelingCnnPolicy(CnnPolicy):
    """CnnPolicy that builds a DuelingQNetwork instead of the standard QNetwork."""

    def make_q_net(self) -> DuelingQNetwork:
        # _update_features_extractor builds the net_args dict that QNetwork expects
        net_args = self._update_features_extractor(
            self.net_args, features_extractor=None
        )
        return DuelingQNetwork(**net_args).to(self.device)

In [ ]:
# ---------------------------------------------------------------------------
# Dueling Double DQN
# Combines the dueling policy with the Double DQN update rule.
# ---------------------------------------------------------------------------

class DuelingDoubleDQN(DQN):
    """
    DQN with:
      - Dueling Q-network head (V + A streams)
      - Double DQN target update (online net selects, target net evaluates)
    """

    def __init__(self, *args, **kwargs):
        kwargs["policy"] = DuelingCnnPolicy
        super().__init__(*args, **kwargs)

    def train(self, gradient_steps: int, batch_size: int = 100) -> None:
        self.policy.set_training_mode(True)
        self._update_learning_rate(self.policy.optimizer)

        losses = []
        for _ in range(gradient_steps):
            replay_data = self.replay_buffer.sample(batch_size, env=self._vec_normalize_env)
            discounts = replay_data.discounts if replay_data.discounts is not None else self.gamma

            with th.no_grad():
                # Double DQN: online net selects action, target net scores it
                next_actions  = self.q_net(replay_data.next_observations).argmax(dim=1, keepdim=True)
                next_q_values = self.q_net_target(replay_data.next_observations).gather(1, next_actions)
                next_q_values = next_q_values.reshape(-1, 1)
                target_q_values = replay_data.rewards + (1 - replay_data.dones) * discounts * next_q_values

            current_q_values = self.q_net(replay_data.observations)
            current_q_values = th.gather(current_q_values, dim=1, index=replay_data.actions.long())

            loss = F.smooth_l1_loss(current_q_values, target_q_values)
            losses.append(loss.item())

            self.policy.optimizer.zero_grad()
            loss.backward()
            th.nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
            self.policy.optimizer.step()

        self._n_updates += gradient_steps
        self.logger.record("train/n_updates", self._n_updates, exclude="tensorboard")
        self.logger.record("train/loss", np.mean(losses))

In [ ]:
# VideoRecorderCallback
class VideoRecorderCallback(BaseCallback):
    def __init__(self, eval_env: gym.Env, render_freq: int, n_eval_episodes: int = 1, deterministic: bool = True):
        super().__init__()
        self._eval_env       = eval_env
        self._render_freq    = render_freq
        self._n_eval_episodes = n_eval_episodes
        self._deterministic  = deterministic

    def _on_step(self) -> bool:
        if self.n_calls % self._render_freq == 0:
            screens = []
            def grab_screens(_locals: Dict[str, Any], _globals: Dict[str, Any]) -> None:
                screen = self._eval_env.render()
                screens.append(screen)
            evaluate_policy(
                self.model, self._eval_env,
                callback=grab_screens,
                n_eval_episodes=self._n_eval_episodes,
                deterministic=self._deterministic,
            )
            frames = np.array(screens)
            wandb.log(
                {"trajectory/video": wandb.Video(frames, fps=30, format="mp4")},
                step=self.num_timesteps,
            )
        return True


# HParamCallback
class HParamCallback(BaseCallback):
    def _on_training_start(self) -> None:
        hparam_dict = {
            "algorithm": "DuelingDoubleDQN",
            "policy": "DuelingCnnPolicy",
            "environment": self.model.env.__class__.__name__,
            "buffer_size": self.model.buffer_size,
            "batch_size": self.model.batch_size,
            "tau": self.model.tau,
            "gradient_steps": self.model.gradient_steps,
            "target_update_interval": self.model.target_update_interval,
            "exploration_fraction": self.model.exploration_fraction,
            "exploration_initial_eps": self.model.exploration_initial_eps,
            "exploration_final_eps": self.model.exploration_final_eps,
            "max_grad_norm": self.model.max_grad_norm,
            "seed": self.model.seed,
            "learning rate": self.model.learning_rate,
            "gamma": self.model.gamma,
        }
        metric_dict = {
            "eval/mean_ep_length": 0, "eval/mean_reward": 0,
            "rollout/ep_len_mean": 0, "rollout/ep_rew_mean": 0,
            "rollout/exploration_rate": 0, "time/_episode_num": 0,
            "time/fps": 0, "time/total_timesteps": 0,
            "train/learning_rate": 0.0, "train/loss": 0.0, "train/n_updates": 0.0,
        }
        self.logger.record(
            "hparams", HParam(hparam_dict, metric_dict),
            exclude=("stdout", "log", "json", "csv"),
        )

    def _on_step(self) -> bool:
        return True


# PlotTensorboardValuesCallback
class PlotTensorboardValuesCallback(BaseCallback):
    def __init__(self, eval_env: gym.Env, train_env: gym.Env, model: DQN, verbose=0):
        super().__init__(verbose)
        self._eval_env  = eval_env
        self._train_env = train_env
        self._model     = model

    def _on_training_start(self) -> None:
        output_formats = self.logger.output_formats
        try:
            self.tb_formatter = next(
                f for f in output_formats if isinstance(f, TensorBoardOutputFormat)
            )
        except StopIteration:
            print("TensorBoard formatter not found.")
            return
        self.tb_formatter.writer.add_text("metadata/eval_env", str(self._eval_env.metadata), self.num_timesteps)
        self.tb_formatter.writer.add_text("metadata/train_env", str(self._train_env.metadata), self.num_timesteps)
        self.tb_formatter.writer.add_text("model/q_net", str(self._model.q_net), self.num_timesteps)
        self.tb_formatter.writer.add_text("model/q_net_target", str(self._model.q_net_target), self.num_timesteps)
        self.tb_formatter.writer.flush()

    def _on_step(self) -> bool:
        self.logger.record("time/_episode_num", self.model._episode_num, exclude=("stdout", "log", "json", "csv"))
        self.logger.record("train/n_updates", self.model._n_updates, exclude=("stdout", "log", "json", "csv"))
        self.logger.record("locals/rewards", self.locals["rewards"], exclude=("stdout", "log", "json", "csv"))
        self.logger.record("locals/infos_0_lives", self.locals["infos"][0]["lives"], exclude=("stdout", "log", "json", "csv"))
        self.logger.record("locals/num_collected_steps", self.locals["num_collected_steps"], exclude=("stdout", "log", "json", "csv"))
        self.logger.record("locals/num_collected_episodes", self.locals["num_collected_episodes"], exclude=("stdout", "log", "json", "csv"))
        return True

    def _on_training_end(self) -> None:
        self.tb_formatter.writer.add_text("metadata/eval_env", str(self._eval_env.metadata), self.num_timesteps)
        self.tb_formatter.writer.add_text("metadata/train_env", str(self._train_env.metadata), self.num_timesteps)
        self.tb_formatter.writer.add_text("model/q_net", str(self._model.q_net), self.num_timesteps)
        self.tb_formatter.writer.add_text("model/q_net_target", str(self._model.q_net_target), self.num_timesteps)
        self.tb_formatter.writer.flush()

In [ ]:
# ---------------------------------------------------------------------------
# W&B setup
# ---------------------------------------------------------------------------

wandb.login()

run = wandb.init(
    project="ale-pacman-v5",
    name="dueling-double-dqn-v1",
    config={
        "algorithm": "DuelingDoubleDQN",
        "total_timesteps": NUM_TIMESTEPS,
        "learning_rate": LEARNING_RATE,
        "buffer_size": BUFFER_SIZE,
        "batch_size": BATCH_SIZE,
        "gamma": GAMMA,
        "exploration_fraction": EXPLORATION_FRACTION,
        "exploration_final_eps": FINAL_EPSILON,
        "target_update_interval": TARGET_UPDATE_INTERVAL,
        "trained_from": "scratch",
    }, 
    id="ykmc0k4e",
    sync_tensorboard=True,
    resume="allow",  # allows resuming the same W&B run after a disconnect
)
print(f"W&B run: {wandb.run.url}")

In [ ]:
# ---------------------------------------------------------------------------
# Environment + Model
# Auto-resumes from the latest checkpoint in SAVE_DIR if one exists.
# ---------------------------------------------------------------------------

eval_env  = Monitor(gym.make("ALE/Pacman-v5", render_mode="rgb_array", frameskip=FRAMESKIP))
train_env = gym.make("ALE/Pacman-v5", render_mode="rgb_array", frameskip=FRAMESKIP)

CUSTOM_OBJECTS = {
    "exploration_fraction": EXPLORATION_FRACTION,
    "buffer_size": BUFFER_SIZE,
    "batch_size": BATCH_SIZE,
    "learning_starts": LEARNING_STARTS,
    "learning_rate": LEARNING_RATE,
    "gamma": GAMMA,
    "target_update_interval": TARGET_UPDATE_INTERVAL,
    "exploration_final_eps": FINAL_EPSILON,
    "tensorboard_log": SAVE_DIR,
    "verbose": 1,
}

# Find the latest checkpoint saved by CheckpointCallback
# Checkpoints are named: rl_model_{timesteps}_steps.zip
checkpoints = sorted(
    glob.glob(os.path.join(SAVE_DIR, "checkpoints", "rl_model_*_steps.zip")),
    key=os.path.getmtime,
)

resuming_from_checkpoint = bool(checkpoints)

if checkpoints:
    latest = checkpoints[-1]
    print(f"Resuming from checkpoint: {latest}")
    model = DuelingDoubleDQN.load(
        latest,
        env=train_env,
        custom_objects=CUSTOM_OBJECTS,
    )
else:
    print("No checkpoint found — training from scratch")
    model = DuelingDoubleDQN(
        policy="CnnPolicy",  # overridden internally to DuelingCnnPolicy
        env=train_env,
        learning_rate=LEARNING_RATE,
        buffer_size=BUFFER_SIZE,
        batch_size=BATCH_SIZE,
        learning_starts=LEARNING_STARTS,
        gamma=GAMMA,
        exploration_fraction=EXPLORATION_FRACTION,
        exploration_final_eps=FINAL_EPSILON,
        target_update_interval=TARGET_UPDATE_INTERVAL,
        tensorboard_log=SAVE_DIR,
        verbose=1,
        seed=42,
    )

print(model.q_net)

In [ ]:
# ---------------------------------------------------------------------------
# Callbacks
# ---------------------------------------------------------------------------

# Saves rl_model_{timesteps}_steps.zip to Drive every CHECKPOINT_FREQ steps
checkpoint_callback = CheckpointCallback(
    save_freq=CHECKPOINT_FREQ,
    save_path=os.path.join(SAVE_DIR, "checkpoints"),
    name_prefix="rl_model",
    save_replay_buffer=False,  # replay buffer is ~14GB, skip for checkpoints
    save_vecnormalize=False,
)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=os.path.join(SAVE_DIR, "best_model"),
    log_path=os.path.join(SAVE_DIR, "evals"),
    eval_freq=EVAL_CALLBACK_FREQ,
    n_eval_episodes=10,
    deterministic=True,
    render=False,
)

tbplot_callback = PlotTensorboardValuesCallback(eval_env=eval_env, train_env=train_env, model=model)
video_callback  = VideoRecorderCallback(eval_env, render_freq=VIDEO_CALLBACK_FREQ)
hparam_callback = HParamCallback()

callback_list = CallbackList([
    hparam_callback,
    checkpoint_callback,
    eval_callback,
    video_callback,
    tbplot_callback,
])

In [ ]:
# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------

# reset_num_timesteps=True so timestep count is preserved across resumes.
# On the very first run (no checkpoint) this starts from 0.
# After a resume it continues from where it left off.
model.learn(
    total_timesteps=NUM_TIMESTEPS,
    callback=callback_list,
    tb_log_name="dueling_double_dqn",
    reset_num_timesteps=not resuming_from_checkpoint,
)

run.finish()

In [ ]:
# ---------------------------------------------------------------------------
# Save final model to Drive
# ---------------------------------------------------------------------------

model.save(MODEL_FILE_NAME)
model.save_replay_buffer(BUFFER_FILE_NAME)
model.policy.save(POLICY_FILE_NAME)

print(f"Saved to {SAVE_DIR}")